# 预期ST因子测试

因子定义：连续两年净利润为负（老规则）

策略逻辑：
1. 每天从全市场筛选总市值最小的500只股票（尾部500）
2. 在尾部500中，筛选连续两年净利润为负的股票
3. 等权持有所有满足条件的股票
4. 每天调仓

In [ ]:
# 第一步：查看表中有哪些可用字段
import dai

# 查看 cn_stock_prefactors_community 表的字段
df = dai.query("SELECT * FROM cn_stock_prefactors_community LIMIT 1", filters={"date": ["2024-01-01", "2024-01-10"]}).df()
print("可用字段：")
for col in sorted(df.columns):
    print(f"  {col}")

In [ ]:
from bigmodule import M, I
import dai
import pandas as pd


# @param(id="m5", name="initialize")
def m5_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


# @param(id="m5", name="before_trading_start")
def m5_before_trading_start_bigquant_run(context, data):
    pass


# @param(id="m5", name="handle_tick")
def m5_handle_tick_bigquant_run(context, tick):
    pass


# @param(id="m5", name="handle_data")
def m5_handle_data_bigquant_run(context, data):
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    target_instruments = set(today_df["instrument"])
    holding_instruments = set(context.get_account_positions().keys())

    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)

    for i, x in today_df.iterrows():
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        context.order_target_percent(x.instrument, position)


# @param(id="m5", name="handle_trade")
def m5_handle_trade_bigquant_run(context, trade):
    pass


# @param(id="m5", name="handle_order")
def m5_handle_order_bigquant_run(context, order):
    pass


# @param(id="m5", name="after_trading")
def m5_after_trading_bigquant_run(context, data):
    pass


# ========== 数据准备 ==========
# TODO: 根据第一个cell的输出，确认正确的字段名后修改下面的SQL
# 需要找到：总市值字段、净利润字段

stock_sql = """
SELECT
    date,
    instrument,
    float_market_cap,  -- 流通市值，需确认总市值字段名
    -- TODO: 添加净利润字段
FROM cn_stock_prefactors_community
WHERE
    st_status = 0
    AND suspended = 0
    AND is_bz50 = 0
    AND list_days > 365
QUALIFY
    ROW_NUMBER() OVER (PARTITION BY date ORDER BY float_market_cap ASC) <= 500
"""

print("正在查询数据...")
stock_data = dai.query(stock_sql, filters={"date": ["2020-01-01", "2026-12-31"]}).df()
print(f"查询完成，共 {len(stock_data)} 条记录")
stock_data.head()

In [ ]:
# ========== 筛选预期ST条件 + 回测 ==========
# TODO: 根据实际字段名完善筛选逻辑

# 筛选：连续两年净利润为负
# filtered_df = stock_data[
#     (stock_data['净利润字段'] < 0) & 
#     (stock_data['去年净利润字段'] < 0)
# ].copy()

# 暂时用全部数据测试流程
filtered_df = stock_data.copy()
print(f"满足条件的记录数：{len(filtered_df)}")

# 计算每日持仓数量并分配等权仓位
daily_count = filtered_df.groupby('date')['instrument'].transform('count')
filtered_df['position'] = 1.0 / daily_count
filtered_df['score'] = -filtered_df['float_market_cap']
filtered_df['score_rank'] = filtered_df.groupby('date')['float_market_cap'].rank(ascending=True).astype(int)

print(f"每日平均持仓数量：{daily_count.groupby(filtered_df['date']).first().mean():.1f}")

stock_data_ds = dai.DataSource.write_bdb(filtered_df)

# ========== 回测 ==========
start_date = '2021-01-01'
end_date = '2026-04-07'

m5 = M.bigtrader.v30(
    data=stock_data_ds,
    start_date=start_date,
    end_date=end_date,
    initialize=m5_initialize_bigquant_run,
    before_trading_start=m5_before_trading_start_bigquant_run,
    handle_tick=m5_handle_tick_bigquant_run,
    handle_data=m5_handle_data_bigquant_run,
    handle_trade=m5_handle_trade_bigquant_run,
    handle_order=m5_handle_order_bigquant_run,
    after_trading=m5_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="daily",
    product_type="股票",
    rebalance_period_type="交易日",
    rebalance_period_days="1",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="标准模式",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="open",
    order_price_field_sell="open",
    benchmark="沪深300指数",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="m5"
)

In [ ]:
# ========== 导出交易记录CSV ==========
trades_df = m5.raw_perf.read()['transactions']

output_records = []
holdings = {}

for idx, row in trades_df.iterrows():
    instrument = row['symbol']
    dt = pd.to_datetime(row['dt']).strftime('%Y-%m-%d')
    amount = row['amount']
    price = row['price']
    
    if amount > 0:
        holdings[instrument] = {'buy_date': dt, 'buy_price': price}
    elif amount < 0 and instrument in holdings:
        buy_info = holdings.pop(instrument)
        pnl = (price - buy_info['buy_price']) / buy_info['buy_price']
        output_records.append({
            '股票代码': instrument.split('.')[0],
            '买入日期': buy_info['buy_date'],
            '卖出日期': dt,
            '买入价格(前复权)': round(buy_info['buy_price'], 2),
            '卖出价格(前复权)': round(price, 2),
            '涨幅': round(pnl, 4)
        })

output_df = pd.DataFrame(output_records)
output_df = output_df.sort_values('卖出日期', ascending=False)

output_path = './strategy/预期ST_bigquant交易记录.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"交易记录已保存到：{output_path}")
print(f"共 {len(output_df)} 条交易记录")
output_df.head(20)